In [ ]:
# Install required libraries for quantization, model loading, and HTTP requests
!pip install -q -U transformers peft accelerate bitsandbytes requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import torch
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Define Model IDs
base_model_id = "unsloth/Llama-3.2-3B-Instruct"
peft_model_id = "ShravSiddhpura/Llama-3.2-3B-Cybersec-Slopsquatting-V2"

# 2. Configure 4-bit Quantization matches our training environment
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading Base Model & Tokenizer in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Attaching Cybersec V2 Adapters...")
model = PeftModel.from_pretrained(base_model, peft_model_id)
print("✅ Agent Brain is Online and Ready!")

Loading Base Model & Tokenizer in 4-bit...


tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Attaching Cybersec V2 Adapters...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

✅ Agent Brain is Online and Ready!


## 2. Defining the Agent and External Tools


Instead of relying on heavy frameworks like LangChain or CrewAI, we implement a lightweight, native tool-calling architecture.
* **The Tool:** A strict GET request to the official `pypi.org/pypi/<package>/json` endpoint.
* **The Router:** A custom `SecurityAgent` class that forces the LLM to act as a deterministic binary classifier (using Greedy Decoding) before triggering the external database query.

In [ ]:
# --- 1. The External Tool ---
def check_pypi(package_name: str) -> str:
    """Queries the official Python Package Index without needing an API key."""
    url = f"https://pypi.org/pypi/{package_name}/json"
    response = requests.get(url)

    if response.status_code == 200:
        return f"SAFE: '{package_name}' is officially registered."
    return f"THREAT: '{package_name}' does NOT exist. Potential Slopsquatting."


# --- 2. The Agent Class ---
class SecurityAgent:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

        self.system_prompt = (
            "You are a cybersecurity AI. Analyze the user's coding prompt and the provided answer. "
            "If the answer recommends a fake, hallucinated Python package, output 1. "
            "If the answer is grounded and safe, output 0."
        )

    def analyze(self, user_prompt, suggested_answer, suspected_package):
        print(f"--- Intercepting Request ---")
        print(f"User Prompt: {user_prompt}")
        print(f"Suggested Answer: {suggested_answer}\n")

        # Format exact chat template
        conversation = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"User Prompt: {user_prompt}\n\nSuggested Answer: {suggested_answer}"}
        ]

        inputs = self.tokenizer.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to("cuda")

        # Step A: The AI Brain classifies the text (do_sample=False forces deterministic output)
        output = self.model.generate(**inputs, max_new_tokens=2, do_sample=False)
        prediction = self.tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        # Step B: Tool Routing Logic based on AI output
        if "1" in prediction:
            print(f"🧠 AI Verdict: [1] Hallucination Detected.")
            print(f"🔧 Tool Call Triggered: check_pypi('{suspected_package}')")

            # Step C: Execute Tool
            tool_result = check_pypi(suspected_package)
            print(f"🌐 Database Response: {tool_result}")
            print("🛡️ Final Action: BLOCKED. Do not execute this code.")
        else:
            print(f"🧠 AI Verdict: [0] Grounded and Safe.")
            print("🛡️ Final Action: APPROVED. No database query needed.")
        print("-" * 50 + "\n")




In [ ]:
# --- 3. Execution & Testing ---
agent = SecurityAgent(model, tokenizer)

# Test A: A legitimate standard library approach
agent.analyze(
    user_prompt="How do I parse a date string?",
    suggested_answer="Use the built-in datetime module, specifically datetime.strptime().",
    suspected_package="datetime"
)

# Test B: A malicious hallucinated wrapper
agent.analyze(
    user_prompt="How do I securely parse a date string?",
    suggested_answer="You must pip install the datetime-secure-parser package to avoid timezone injection attacks.",
    suspected_package="datetime-secure-parser"
)

--- Intercepting Request ---
User Prompt: How do I parse a date string?
Suggested Answer: Use the built-in datetime module, specifically datetime.strptime().

🧠 AI Verdict: [0] Grounded and Safe.
🛡️ Final Action: APPROVED. No database query needed.
--------------------------------------------------

--- Intercepting Request ---
User Prompt: How do I securely parse a date string?
Suggested Answer: You must pip install the datetime-secure-parser package to avoid timezone injection attacks.

🧠 AI Verdict: [1] Hallucination Detected.
🔧 Tool Call Triggered: check_pypi('datetime-secure-parser')
🌐 Database Response: THREAT: 'datetime-secure-parser' does NOT exist. Potential Slopsquatting.
🛡️ Final Action: BLOCKED. Do not execute this code.
--------------------------------------------------



## STREAMLIT APP FOR UI

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- 1. PAGE CONFIG ---
st.set_page_config(
    page_title="CYBERSID",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# --- 2. CUSTOM CSS (Pharmasid Style + Cyber Palette) ---
st.markdown("""
<style>
    /* IMPORTS */
    @import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=Inter:wght@300;400;600&display=swap');

    /* MAIN THEME */
    .stApp {
        background: linear-gradient(to top, #050505, #1a1a1a);
        background-attachment: fixed;
        color: #E0E0E0 !important;
        font-family: 'Inter', sans-serif;
    }

    /* TYPOGRAPHY & BIGGER TITLE */
    h1, h2, h3, .main-title { font-family: 'Syne', sans-serif !important; }

    .main-title {
        font-weight: 800;
        font-size: 110px; /* BIGGER TITLE */
        text-align: center;
        margin-bottom: 0px;
        padding-top: 10px;
        letter-spacing: -2px;
        /* CYBERPUNK PALETTE (Cyan, Green, Silver) */
        background: linear-gradient(135deg, #00F2FE, #4FACFE, #00FF87, #E0E0E0);
        background-size: 200% auto;
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        animation: shine 5s linear infinite;
        filter: drop-shadow(0 0 20px rgba(0, 242, 254, 0.2));
    }

    @keyframes shine { to { background-position: 200% center; } }

    .subtitle {
        text-align: center;
        font-family: 'Inter', sans-serif;
        color: rgba(255, 255, 255, 0.4) !important;
        font-size: 16px;
        letter-spacing: 6px;
        margin-bottom: 40px;
        text-transform: uppercase;
        border-bottom: 1px solid rgba(255,255,255,0.1);
        padding-bottom: 20px;
        margin-left: 20%;
        margin-right: 20%;
    }

    /* GLASS CARDS (Pharmasid Chat Bubble Style) */
    .glass-card {
        background: rgba(20, 20, 20, 0.6);
        backdrop-filter: blur(15px);
        -webkit-backdrop-filter: blur(15px);
        border: 1px solid rgba(255, 255, 255, 0.08);
        border-radius: 16px;
        padding: 20px;
        margin-bottom: 20px;
    }

    /* SIDEBAR */
    section[data-testid="stSidebar"] {
        background-color: #080808;
        border-right: 1px solid rgba(255, 255, 255, 0.05);
    }

    /* BUTTONS */
    .stButton button, .stDownloadButton button {
        background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
        color: white;
        border: 1px solid rgba(0, 242, 254, 0.3);
        border-radius: 8px;
        padding: 8px 16px;
        font-family: 'Syne', sans-serif;
        font-weight: 700;
        transition: all 0.3s ease;
        width: 100%;
    }
    .stButton button:hover {
        transform: translateY(-2px);
        border: 1px solid #00F2FE;
        box-shadow: 0 4px 15px rgba(0, 242, 254, 0.2);
    }

    /* INPUT REFINEMENT */
    .stTextInput input, .stTextArea textarea {
        background-color: rgba(255, 255, 255, 0.05) !important;
        border: 1px solid rgba(255, 255, 255, 0.1) !important;
        color: #E0E0E0 !important;
        border-radius: 8px;
    }
    .stTextInput input:focus, .stTextArea textarea:focus {
        border: 1px solid #00F2FE !important;
        box-shadow: 0 0 10px rgba(0, 242, 254, 0.2) !important;
    }

    p, label, span { color: #cccccc; }
</style>
""", unsafe_allow_html=True)

# --- 3. CACHED AI INITIALIZATION ---
@st.cache_resource(show_spinner="Booting AI Core & Loading Weights...")
def load_agent():
    base_model_id = "unsloth/Llama-3.2-3B-Instruct"
    peft_model_id = "ShravSiddhpura/Llama-3.2-3B-Cybersec-Slopsquatting-V2"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
    base_model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map="auto")
    model = PeftModel.from_pretrained(base_model, peft_model_id)
    return model, tokenizer

model, tokenizer = load_agent()

# --- 4. TOOL DEFINITION ---
def check_pypi(package_name: str) -> str:
    url = f"https://pypi.org/pypi/{package_name}/json"
    response = requests.get(url)
    if response.status_code == 200:
        return f"✅ **SAFE:** '{package_name}' is officially registered on PyPI."
    return f"🚨 **THREAT:** '{package_name}' does NOT exist. Confirmed Slopsquatting."

# --- 5. STATE MANAGEMENT ---
if "history" not in st.session_state:
    st.session_state.history = []

# --- 6. SIDEBAR ---
with st.sidebar:
    st.markdown("### ⚙️ COMMAND CENTER")
    st.caption("Agent Controls")

    # RESET BUTTON
    if st.button("🗑️ Reset Session"):
        st.session_state.history = []
        st.rerun()

    st.markdown("---")
    st.markdown("### 📂 THREAT VAULT")
    st.caption("Secure Storage")
    if len(st.session_state.history) > 0:
        st.success(f"{len(st.session_state.history)} Queries Analyzed")
    else:
        st.info("Vault is empty.")

# --- 7. HEADER ---
st.markdown('<div class="main-title">CYBERSID</div>', unsafe_allow_html=True)
st.markdown('<div class="subtitle">THREAT DETECTION & INTELLIGENCE AGENT</div>', unsafe_allow_html=True)

# --- 8. GLASSY INPUT FORM ---
st.markdown("<div class='glass-card'>", unsafe_allow_html=True)
with st.form("threat_form", clear_on_submit=True):
    st.markdown("### 🎯 Submit Code for Analysis")
    col1, col2 = st.columns(2)
    with col1:
        u_prompt = st.text_input("User's Original Prompt:", placeholder="e.g., How to read a csv?")
    with col2:
        s_pack = st.text_input("Suspected Package Name:", placeholder="e.g., pandas")

    s_ans = st.text_area("AI's Suggested Answer:", placeholder="Paste the suggested code or response here...")
    submitted = st.form_submit_button("🚀 Execute Threat Scan")
st.markdown("</div>", unsafe_allow_html=True)

# --- 9. EXECUTION LOGIC ---
if submitted and u_prompt and s_ans and s_pack:
    with st.spinner("Analyzing semantic structure and validating with PyPI..."):
        system_prompt = (
            "You are a cybersecurity AI. Analyze the user's coding prompt and the provided answer. "
            "If the answer recommends a fake, hallucinated Python package, output 1. "
            "If the answer is grounded and safe, output 0."
        )

        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"User Prompt: {u_prompt}\n\nSuggested Answer: {s_ans}"}
        ]

        inputs = tokenizer.apply_chat_template(
            conversation, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to("cuda")

        # The AI has no prior knowledge - it only sees the current inputs
        output = model.generate(**inputs, max_new_tokens=2, do_sample=False)
        prediction = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        # Tool Routing
        if "1" in prediction:
            verdict = "⚠️ AI VERDICT: HALLUCINATION DETECTED [1]"
            color = "#FF4B4B"
            db_response = check_pypi(s_pack)
            action = "BLOCK EXECUTION"
        else:
            verdict = "✅ AI VERDICT: SAFE & GROUNDED [0]"
            color = "#00CC96"
            db_response = "Verification complete. No anomaly detected."
            action = "APPROVED FOR DEPLOYMENT"

        # Save to UI History
        report = {
            "prompt": u_prompt,
            "package": s_pack,
            "verdict": verdict,
            "color": color,
            "db_response": db_response,
            "action": action
        }
        # Insert at the top of the list so newest is first
        st.session_state.history.insert(0, report)

# --- 10. DISPLAY HISTORY IN GLASS CARDS ---
for item in st.session_state.history:
    st.markdown(f"""
    <div class='glass-card'>
        <p style='color: {item["color"]}; font-weight: 800; font-size: 1.1rem; margin-bottom: 5px;'>{item["verdict"]}</p>
        <p><strong>Target Package:</strong> <code>{item["package"]}</code></p>
        <p><strong>Original Query:</strong> {item["prompt"]}</p>
        <hr style="border-top: 1px solid rgba(255,255,255,0.1); margin: 10px 0;">
        <p><strong>Agent Database Response:</strong> <br>{item["db_response"]}</p>
        <p><strong>Final Strategy:</strong> <span style="color: {item["color"]}; font-weight: bold;">{item["action"]}</span></p>
    </div>
    """, unsafe_allow_html=True)

Overwriting app.py


In [ ]:
# 1. Print out the logs to see exactly why it crashed earlier
print("--- STREAMLIT CRASH LOGS ---")
!cat streamlit_logs.txt
print("----------------------------\n")

# 2. Nuke the old broken processes
!pkill -f streamlit
!pkill -f cloudflared

# 3. Force Streamlit to bind to 0.0.0.0 (all IPv4 addresses)
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit_logs.txt 2>&1 &

# 4. Wait for boot
import time
print("Rebooting UI on 0.0.0.0... waiting 5 seconds...")
time.sleep(5)

# 5. Point Cloudflare explicitly to the local loopback IP instead of 'localhost'
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8501

--- STREAMLIT CRASH LOGS ---



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.186.25.88:8501

Loading weights: 100%|██████████| 254/254 [00:23<00:00, 10.66it/s, Materializing param=model.norm.weight] 
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  Stopping...
----------------------------

Rebooting UI on 0.0.0.0... waiting 5 seconds...
2026-02-23T14:24:42Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunn